In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("adult.csv")

# Limpeza
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df.replace("?", np.nan, inplace=True)
df.dropna(inplace=True)

In [2]:
#criar variavel sem faculdade
college_levels = [
    "Bachelors",
    "Masters",
    "Doctorate",
    "Prof-school",
    "Assoc-acdm",
    "Assoc-voc"
]

df["no_college"] = ~df["education"].isin(college_levels)

In [3]:
#Transformar Target
df["income"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})

In [4]:
#Filtrar Apenas Quem NÃO Tem Faculdade
df_no_college = df[df["no_college"] == True]

In [5]:
#Calcular Média por Sexo
resultado = (
    df_no_college
    .groupby("sex")["income"]
    .mean()
    .reset_index()
)


In [6]:
#Regressão Apenas com Quem Não Tem Faculdade
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Filtrar
df_model = df[df["no_college"] == True].copy()

X = df_model[["sex"]]
y = df_model["income"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"), ["sex"])
    ]
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("regressor", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

pipeline.fit(X_train, y_train)

coef = pipeline.named_steps["regressor"].coef_
intercept = pipeline.named_steps["regressor"].intercept_

print("Intercept:", intercept)
print("Coeficiente sexo (Male):", coef)

Intercept: 0.06058921623123975
Coeficiente sexo (Male): [0.14616379]


In [7]:
# CARREGAR E LIMPAR DATASET
df = pd.read_csv("adult.csv")

df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df.replace("?", np.nan, inplace=True)
df.dropna(inplace=True)


In [8]:
#DEFINIR QUEM TEM FACULDADE
college_levels = [
    "Bachelors",
    "Masters",
    "Doctorate",
    "Prof-school",
    "Assoc-acdm",
    "Assoc-voc"
]

df["no_college"] = ~df["education"].isin(college_levels)



In [9]:
#TRANSFORMAR TARGET
df["income"] = df["income"].map({
    "<=50K": 0,
    ">50K": 1
})


In [10]:
#FILTRAR QUEM NÃO TEM FACULDADE
df_no_college = df[df["no_college"] == True]



In [11]:
# % QUE GANHAM >50K (ENTRE SEM FACULDADE)
percent_high_income = df_no_college["income"].mean() * 100



In [12]:
#HOMEM E MULHER DENTRO DO GRUPO QUE GANHA >50K
df_high_income = df_no_college[df_no_college["income"] == 1]

gender_distribution = (
    df_high_income["sex"]
    .value_counts(normalize=True) * 100
)



In [13]:
#RESULTADOS
print("===== RESULTADOS =====\n")

print(f"Percentual de pessoas SEM faculdade que ganham >50K: {percent_high_income:.2f}%\n")

for gender, value in gender_distribution.items():
    print(f"Percentual de {gender} entre os que ganham >50K e não têm faculdade: {value:.2f}%")

===== RESULTADOS =====

Percentual de pessoas SEM faculdade que ganham >50K: 15.69%

Percentual de Male entre os que ganham >50K e não têm faculdade: 86.85%
Percentual de Female entre os que ganham >50K e não têm faculdade: 13.15%
